In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import json
from itertools import islice
from river import metrics, tree, drift
 
from ensemble import DriftAdaptiveEnsemble
from centroid_drift import CentroidDriftDetector
from transformer import FeatureDistrict
from river import metrics, dummy, forest, tree, stats, ensemble, drift
from utils import warsaw_stream, airline_stream, taxi_stream
from river.forest import ARFRegressor
from river.tree import HoeffdingTreeRegressor
from river.drift import ADWIN, PageHinkley
from river import metrics
import copy
from collections import defaultdict
from itertools import islice

import os

In [2]:
# %pip install -r requirements.txt

In [3]:
DATASET = "Airplanes"

In [4]:
df_warsaw=pd.read_csv("dataset/warsaw_synthetic.csv")
df_airplanes=pd.read_csv("dataset/Airplanes_modified.csv")
df_taxi=pd.read_csv("dataset/taxi_dataset_ordered.csv")

if DATASET=="Airplanes":
    rows = df_airplanes.to_dict(orient='records')
    data = airline_stream(rows)
    leng=df_airplanes.shape[0]
    size=df_airplanes.shape[1]
elif DATASET=="Warsaw":
    rows = df_warsaw.to_dict(orient='records')
    data = warsaw_stream(rows)
    leng=len(df_warsaw)
elif DATASET=="Taxi":
    rows = df_taxi.to_dict(orient='records')
    data = taxi_stream(rows)
    leng=len(df_taxi)
else:
    raise Exception("Dataset ERROR")
print(leng)
print(size)

2319121
17


In [5]:
MAX_SAMPLES = leng
seed = 17
 
with open("transformer/config_dataset.json", "r", encoding="utf-8-sig") as f:
    config = json.load(f)
cfg = config[DATASET]

In [6]:
districts = cfg["district_list"]

columns_to_drop = cfg["columns_to_drop"]

WARMUP_DAYS = cfg["WARMUP_DAYS"]
PH_THRESHOLD = cfg["PH_THRESHOLD"]
PH_MIN_INSTANCES = cfg["PH_MIN_INSTANCES"]
PH_ALPHA = cfg["PH_ALPHA"]

In [7]:
DISTRICTS_SETTINGS=cfg["DISTRICTS_SETTINGS"]

In [8]:
transformer = FeatureDistrict(
    dataset=DATASET,
    columns_to_drop=cfg["columns_to_drop"],
)

In [9]:
data = islice(data, MAX_SAMPLES)

In [10]:
def make_models(districts, district_settings, base_estimator, metric, **ensemble_kwargs):
    return {
        district: DriftAdaptiveEnsemble(
            base_estimator=copy.deepcopy(base_estimator),
            drift_detector= drift.ADWIN(delta=DISTRICTS_SETTINGS[district]["delta_c"], clock=DISTRICTS_SETTINGS[district]["window"]),
            warning_detector=drift.ADWIN(delta=DISTRICTS_SETTINGS[district]["delta_w"]),
            metric=copy.deepcopy(metric),
            **ensemble_kwargs,
        )
        for district in districts
    }





In [11]:
# DISTRICTS_SETTINGS = {
#     "South": {"delta_w": 0.7, "delta_c": 0.5, "window": 2000},    
#     "West": {"delta_w": 0.7, "delta_c": 0.5, "window": 2000},         
#     "Midwest": {"delta_w": 0.7, "delta_c":  0.5, "window": 300},    
#     "Northeast": {"delta_w": 0.5, "delta_c": 0.2, "window": 200},    
#     "Global": {"delta_w": 0.5, "delta_c": 0.1, "window": 1000}      
# }

In [12]:
rmse_metric = metrics.RMSE()

feature_drift_detectors = {d: CentroidDriftDetector(
    warmup_days=WARMUP_DAYS,
    ph_threshold=PH_THRESHOLD,
    ph_min_instances=PH_MIN_INSTANCES,
    ph_alpha=PH_ALPHA,
) for d in districts}

In [13]:
#m=HoeffdingTreeRegressor()
m=ensemble.SRPRegressor(n_models=3, seed=17)
models = make_models(districts, DISTRICTS_SETTINGS, m, metrics.RMSE(), max_ensemble_size=4, retain_initial_model=True)


In [14]:
records = []
drift_records = []
event_offsets = {k: 0 for k in models.keys()}
local_counts  = {d: 0 for d in districts}

feature_drift_records=[]

In [15]:
for i, (x_raw, y) in enumerate(tqdm(data, total=MAX_SAMPLES)):
    
    x = transformer.transform_one(x_raw)
    timestamp = x['timestamp']
    x.pop("timestamp")
    district = (
            x["pickup_district"]
            if x["within_district"] == 1
            else "Global"
        )
    x.pop("pickup_district", None)
    x.pop("dropoff_district", None)
    local_counts[district] += 1
    instance_count = local_counts[district]

    model = models[district]
    y_hat = model.predict_one(x)
    model.learn_one(x, y, timestamp)

    records.append({
            "n_seen":         i,
            "timestamp": timestamp,
            "district":  district,
            "y":         y,
            "y_hat":     y_hat,
        })

    
    log = model.drift_log
    offset = event_offsets[district]
    if len(log) > offset:
        for _, event in log.iloc[offset:].iterrows():
            drift_records.append({**event, "district": district})
        event_offsets[district] = len(log)
 
 
    feature_vec = np.array(list(x.values()), dtype=float)
    feature_drift_detectors[district].update(feature_vec, timestamp, instance_count)
 
    if feature_drift_detectors[district].drift_detected:
        last = feature_drift_detectors[district].drift_log[-1]
        last["district"] = district
        last["event_type"]="centroid"
        feature_drift_records.append(last)
        

100%|██████████| 2319121/2319121 [5:21:33<00:00, 120.20it/s]  


In [16]:
df_events=pd.DataFrame(drift_records)
df_events_feature=pd.DataFrame(feature_drift_records)

df_events.to_csv(f"results_drift/results_raw/{DATASET}_events_2.csv", index=False)
df_events_feature.to_csv(f"results_drift/results_raw/{DATASET}_events_centroid_2.csv", index=False)

df_predictions = pd.DataFrame(records)


In [17]:
df_events_2 = df_events[
    (df_events["event_type"] == "drift") |
    (df_events["event_type"] == "warning")
]
df_events_2=df_events_2[["timestamp","n_seen","event_type","district"]]
df_centroid=df_events_feature[["date","instance","event_type","district"]]
df_centroid = df_centroid.rename(columns={
    "date": "timestamp",
    "instance": "n_seen"
})
drift_log_final=pd.concat([df_events_2, df_centroid], axis=0)
df_events_2.head(5)


,timestamp,n_seen,event_type,district
0,2008-01-01 08:00:00,1594,warning,Global
1,2008-01-01 11:00:00,3498,drift,Global
3,2008-01-01 12:00:00,3626,warning,Global
4,2008-01-01 12:00:00,1722,warning,West
5,2008-01-01 12:00:00,1658,warning,South


In [18]:
df_predictions.to_csv(f"results_drift/{DATASET}_predictions_2.csv", index=False)
drift_log_final.to_csv(f"results_drift/{DATASET}_drift_log_2.csv", index=False)


In [19]:

# drift_centroid = pd.DataFrame(feature_drift_detectors[district].drift_log)

# drift_centroid2 = drift_centroid.drop(columns=["daily_centroid"], errors="ignore")

# drift_centroid2.to_csv(f"{DATASET}_centroid_drift_log.csv", index=False)